In [1]:
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
csv_path = r"C:\Users\hp\DC\USvideos.csv"
json_path = r"C:\Users\hp\DC\US_category_id.json"

In [4]:
df = pd.read_csv(csv_path, encoding='latin-1')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (40949, 16)

Columns: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description']

First 3 rows:


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John..."
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO â¶ \n\nSUBSCRIBE âº ...


In [5]:
with open(json_path, 'r') as f:
    cat_data = json.load(f)

category_map = {}
for item in cat_data['items']:
    category_map[int(item['id'])] = item['snippet']['title']

print("Categories found:", len(category_map))
print(category_map)

Categories found: 32
{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 18: 'Short Movies', 19: 'Travel & Events', 20: 'Gaming', 21: 'Videoblogging', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 31: 'Anime/Animation', 32: 'Action/Adventure', 33: 'Classics', 34: 'Comedy', 35: 'Documentary', 36: 'Drama', 37: 'Family', 38: 'Foreign', 39: 'Horror', 40: 'Sci-Fi/Fantasy', 41: 'Thriller', 42: 'Shorts', 43: 'Shows', 44: 'Trailers'}


In [6]:
df['category_name'] = df['category_id'].map(category_map).fillna('Unknown')

df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')
df['publish_time'] = pd.to_datetime(df['publish_time'])

df['publish_date'] = df['publish_time'].dt.date
df['publish_hour'] = df['publish_time'].dt.hour
df['publish_day'] = df['publish_time'].dt.day_name()

df['days_to_trend'] = (df['trending_date'] - df['publish_time'].dt.tz_localize(None)).dt.days

df = df.drop(columns=['thumbnail_link', 'description', 'tags'])

before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Duplicates removed: {before - after}")

print("\nMissing values:")
print(df.isnull().sum())

print("\nCleaned shape:", df.shape)
print("\nNew columns:", df.columns.tolist())

Duplicates removed: 48

Missing values:
video_id                  0
trending_date             0
title                     0
channel_title             0
category_id               0
publish_time              0
views                     0
likes                     0
dislikes                  0
comment_count             0
comments_disabled         0
ratings_disabled          0
video_error_or_removed    0
category_name             0
publish_date              0
publish_hour              0
publish_day               0
days_to_trend             0
dtype: int64

Cleaned shape: (40901, 18)

New columns: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'views', 'likes', 'dislikes', 'comment_count', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'category_name', 'publish_date', 'publish_hour', 'publish_day', 'days_to_trend']


In [7]:
df['engagement_rate'] = ((df['likes'] + df['dislikes'] + df['comment_count']) / df['views'] * 100).round(2)

df['like_ratio'] = (df['likes'] / (df['likes'] + df['dislikes']) * 100).round(2)

df['views_category'] = pd.cut(df['views'], 
    bins=[0, 100000, 500000, 1000000, 5000000, float('inf')],
    labels=['<100K', '100K-500K', '500K-1M', '1M-5M', '5M+'])

df['trending_year'] = df['trending_date'].dt.year
df['trending_month'] = df['trending_date'].dt.month_name()

print("✅ New metrics added!")
print("\nSample engagement data:")
print(df[['title', 'views', 'likes', 'engagement_rate', 'like_ratio', 'views_category']].head(3))

✅ New metrics added!

Sample engagement data:
                                               title    views   likes  \
0                 WE WANT TO TALK ABOUT OUR MARRIAGE   748374   57527   
1  The Trump Presidency: Last Week Tonight with J...  2418783   97185   
2  Racist Superman | Rudy Mancuso, King Bach & Le...  3191434  146033   

   engagement_rate  like_ratio views_category  
0            10.22       95.10        500K-1M  
1             4.80       94.05          1M-5M  
2             5.00       96.47          1M-5M  


In [8]:
fact_videos = df[[
    'video_id', 'title', 'channel_title', 'category_id', 'category_name',
    'trending_date', 'publish_date', 'publish_hour', 'publish_day',
    'trending_year', 'trending_month', 'days_to_trend',
    'views', 'likes', 'dislikes', 'comment_count',
    'engagement_rate', 'like_ratio', 'views_category',
    'comments_disabled', 'ratings_disabled', 'video_error_or_removed'
]].copy()

channel_summary = df.groupby('channel_title').agg(
    Total_Videos=('video_id', 'count'),
    Total_Views=('views', 'sum'),
    Avg_Views=('views', 'mean'),
    Total_Likes=('likes', 'sum'),
    Avg_Engagement=('engagement_rate', 'mean')
).reset_index().round(2)
channel_summary = channel_summary.sort_values('Total_Views', ascending=False)

category_summary = df.groupby('category_name').agg(
    Total_Videos=('video_id', 'count'),
    Total_Views=('views', 'sum'),
    Avg_Views=('views', 'mean'),
    Avg_Likes=('likes', 'mean'),
    Avg_Engagement=('engagement_rate', 'mean')
).reset_index().round(2)
category_summary = category_summary.sort_values('Total_Views', ascending=False)

daily_trends = df.groupby('trending_date').agg(
    Total_Videos=('video_id', 'count'),
    Total_Views=('views', 'sum'),
    Avg_Engagement=('engagement_rate', 'mean')
).reset_index().round(2)

output_path = r"C:\Users\hp\DC"

fact_videos.to_csv(output_path + 'yt_fact_videos.csv', index=False)
channel_summary.to_csv(output_path + 'yt_channel_summary.csv', index=False)
category_summary.to_csv(output_path + 'yt_category_summary.csv', index=False)
daily_trends.to_csv(output_path + 'yt_daily_trends.csv', index=False)

print("✅ All CSVs saved!")
print(f"Fact Videos: {len(fact_videos)} rows")
print(f"Channels: {len(channel_summary)} rows")
print(f"Categories: {len(category_summary)} rows")
print(f"Daily Trends: {len(daily_trends)} rows")

✅ All CSVs saved!
Fact Videos: 40901 rows
Channels: 2207 rows
Categories: 16 rows
Daily Trends: 205 rows


In [11]:
df['like_ratio'] = df['like_ratio'].fillna(0)

df['days_to_trend'] = df['days_to_trend'].clip(lower=0)

output_path = r"C:\Users\hp\DC"
df_final = df.copy()

df_final['like_ratio'] = (df_final['likes'] / (df_final['likes'] + df_final['dislikes']).replace(0, 1) * 100).round(2)
df_final['days_to_trend'] = df_final['days_to_trend'].clip(lower=0)

df_final.to_csv(output_path + 'DCyt_fact_videos.csv', index=False)
print("✅ Fixed & saved!")
print(f"like_ratio nulls: {df_final['like_ratio'].isnull().sum()}")
print(f"Negative days_to_trend: {(df_final['days_to_trend'] < 0).sum()}")

✅ Fixed & saved!
like_ratio nulls: 0
Negative days_to_trend: 0
